This notebook demonstrates a Ultrasonic Immersion Testing simulation using SPECFEM2D for non-destructive evaluation (NDE) applications. It includes steps for setting up the environment, running the simulation, and visualizing the results (A-scan and wavefield animation).

 "This notebook compiles and runs SPECFEM2D with **CUDA GPU acceleration**. It **will not work** on a CPU runtime"
  


### 1 Backup Options
You can choose whether to backup the compiled SPECFEM2D binaries to your Google Drive after installation. This can save time in future sessions if you need to re-run the notebook. If you choose not to backup, the compiled binaries will only be available in the temporary Colab environment for the current session.

In [ ]:
BACKUP_TO_DRIVE = False

print(f"Backup to Google Drive set to: {BACKUP_TO_DRIVE}")

In [ ]:
import os
if BACKUP_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

### 2. Install or Restore SPECFEM2D
This cell checks if a compiled version of SPECFEM2D exists on your Drive. If not, it performs a full installation, including MPI and CUDA configuration for GPU acceleration.

In [ ]:
%%bash -s "$BACKUP_TO_DRIVE"

# Capture the backup flag from Python environment
SHOULD_BACKUP=$1

# Define the backup path on the Google Drive
DRIVE_BACKUP="/content/drive/MyDrive/specfem2d_gpu_backup.tar.gz"
GITHUB_REPO="https://github.com/OpenUltrasonics/specfem2d-UT.git"

# =====================================================================
# LOGIC CHECK: Does the compiled backup already exist on Google Drive?
# =====================================================================
if [ -f "$DRIVE_BACKUP" ]; then
    echo "✅ Found pre-compiled SPECFEM2D backup on Google Drive!"
    echo "Extracting to Colab workspace (this takes about 5 seconds)..."

    # Copy the single zipped file and extract it instantly
    cp "$DRIVE_BACKUP" /content/
    tar -xzf /content/specfem2d_gpu_backup.tar.gz

    echo "✅ Extraction complete. You are ready to run simulations!"

else
    echo "⚠️ No backup found (or Drive not mounted). Proceeding with full GPU installation..."

    # 1. Install required system dependencies for Colab
    echo "Installing compilers, MPI libraries, and Lex/Yacc..."
    apt-get update -qq
    apt-get install -y -qq build-essential gfortran openmpi-bin libopenmpi-dev flex libfl-dev bison > /dev/null

    # 2. Clone the OpenUltrasonics repository
    echo "Cloning the OpenUltrasonics repository..."
    git clone "$GITHUB_REPO" specfem2d
    cd specfem2d

    # 3. Configure and Compile with CUDA
    echo "Configuring and compiling SPECFEM2D for GPU..."
    ./configure --with-mpi --with-cuda=cuda10 CUDA_LIB=/usr/local/cuda/lib64 FLAGS_CHECK="-O2 -Wall -Wno-do-subscript -Wno-conversion -Wno-maybe-uninitialized"
    make clean
    make -j2 all

    # =====================================================================
    # 🚨 HEALTH CHECK: Did the compilation actually succeed?
    # =====================================================================
    if [ ! -f "bin/xspecfem2D" ] || [ ! -f "bin/xmeshfem2D" ]; then
        echo "❌ CRITICAL ERROR: Compilation failed! Binaries were not created."
        exit 1
    fi
    echo "✅ Compilation successful! Binaries verified."

    # 4. Verify the CUDA Device Architecture
    echo "=========================================================="
    echo "Verifying CUDA Device Setup..."
    cd utils/GPU_tools/
    nvcc --gpu-architecture=sm_75 -o check_cuda_device check_cuda_device.cu
    ./check_cuda_device
    cd ../../..
    echo "=========================================================="

    # 5. Conditional Backup to Google Drive
    if [ "$SHOULD_BACKUP" = "True" ]; then
        echo "📦 Compressing the compiled software for backup..."
        cd /content/
        tar -czf specfem2d_gpu_backup.tar.gz specfem2d/

        if [ -d "/content/drive/MyDrive" ]; then
            echo "Moving the compressed backup to your Google Drive..."
            cp specfem2d_gpu_backup.tar.gz "$DRIVE_BACKUP"
            echo "✅ Backup saved! Next time you run this notebook, setup will be instant."
        else
            echo "⚠️ Google Drive not mounted. Backup saved to temporary storage only."
        fi
    else
        echo "ℹ️ Backup to Drive skipped as per user setting."
    fi
fi

### 3. Verify CUDA Device
Before running the simulation, we confirm that the GPU is correctly detected and accessible by the CUDA compiler.

In [ ]:
%cd /content/specfem2d/utils/GPU_tools/
! nvcc --gpu-architecture=sm_75 -o check_cuda_device check_cuda_device.cu
! ./check_cuda_device

### 4. Run Pulse-Echo Simulation
We install Gmsh for mesh generation and execute the `run_this_example.sh` script, which handles meshing, solver execution, and initial data processing.

In [ ]:
%%bash

# 1. Install the physical Gmsh engine into the Colab Linux system
echo "Installing system-level Gmsh..."
apt-get install -y -qq gmsh > /dev/null

# 2. Install the required Python libraries quietly
echo "Installing Python dependencies (PyYAML, ObsPy)..."
pip install -q gmsh pyyaml obspy

# 3. Navigate to your specific example directory
cd /content/specfem2d/EXAMPLES/03_Ultrasonic_Immersion_Testing

# 4. Enable GPU in 00_parameters.yaml
echo "Enabling GPU in 00_parameters.yaml..."
sed -i 's/use_gpu: False/use_gpu: True/g' DATA/00_parameters.yaml

# 5. Grant execution permissions to your master script
chmod +x run_this_example.sh

# Fix: Explicitly set USER environment variable for the script
export USER="root"

# 6. Run the entire modular pipeline!
./run_this_example.sh

5. Visualize Results
The following cells display the generated Wavefield Animation.
Note: The wavefield resolution is intentionally kept low to speed up the simulation. For a high-resolution wavefield, set output_wavefield_dumps = .true. in the Par_file 
— be aware this creates very large output files

In [ ]:
from IPython.display import Image, display
import os

gif_path = '/content/specfem2d/EXAMPLES/03_Ultrasonic_Immersion_Testing/Wavefield_Animation.gif'

if os.path.exists(gif_path):
    print("🌊 Displaying Wavefield Propagation:")
    # Display the GIF
    display(Image(filename=gif_path))
else:
    print("❌ GIF not found. Check the script output above.")

### 6. A-Scan Visualization
Finally, we display the A-scan result. This plot shows the amplitude of the reflected ultrasonic signal over time, allowing us to identify echoes from defects or boundaries.

In [ ]:
from IPython.display import Image, display
import os

image_path = '/content/specfem2d/EXAMPLES/03_Ultrasonic_Immersion_Testing/A_Scan_Result.png'

if os.path.exists(image_path):
    print("🖼️ Displaying A-Scan Image:")
    display(Image(filename=image_path))
else:
    print(f"❌ Image not found at {image_path}. Check the path and previous outputs.")